## Setting up env

In [1]:
import os
from dotenv import load_dotenv
from huggingface_hub import login
from datasets import load_dataset, Dataset, DatasetDict
import matplotlib.pyplot as plt
import pandas as pd
from tqdm import tqdm
import time

/home/yhuang/fine_tuning_project/.venv/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
print("HF_HOME:", os.environ.get("HF_HOME"))
print("HF_DATASETS_CACHE:", os.environ.get("HF_DATASETS_CACHE"))
print("HF_HUB_CACHE:", os.environ.get("HF_HUB_CACHE"))

HF_HOME: /scratch-local/yhuang/huggingface_cache
HF_DATASETS_CACHE: /scratch-local/yhuang/huggingface_cache/datasets
HF_HUB_CACHE: /scratch-local/yhuang/huggingface_cache/hub


In [3]:
# environment

load_dotenv(override=True)
# Log in to HuggingFace

#hf_token = os.environ['HF_TOKEN']
#login(hf_token, add_to_git_credential=True)
hf_token = os.getenv("HF_TOKEN")  # 也可用 "HUGGINGFACEHUB_API_TOKEN"
if not hf_token:
    raise RuntimeError("HF_TOKEN is not set in environment/.env")

login(token=hf_token, add_to_git_credential=True)

Note: Environment variable`HF_TOKEN` is set and is the current active token independently from the token you've just configured.


## Experimenting Instruction-Output Pair from 2025 datapoints

In [14]:
ruozhi_punchline = load_dataset("LooksJuicy/ruozhiba-punchline", split="train", trust_remote_code=True)

In [15]:
ruozhi_punchline[0]

{'raw': '和尚逛漫展，化二次缘。',
 'instruction': '和尚去参加漫展了',
 'thought': "可以通过双关语和反转来制造笑点。'化二次缘'是对和尚参加漫展的一种幽默形象的描述，它既可以理解为和尚在漫展中变得更加二次元，也可以理解为和尚在漫展中找到了新的缘分。这种双关语和反转的效果，使得这个笑点既有趣又有深度。",
 'output': '他化二次缘了'}

In [16]:
def simplify(example):
    return {
        "raw": example["raw"],
        "instruction": example["instruction"],
        "output": example["output"]
    }

simplified_ruozhi_training = ruozhi_punchline.map(simplify, remove_columns=ruozhi_punchline.column_names)
print(simplified_ruozhi_training[0])

{'raw': '和尚逛漫展，化二次缘。', 'instruction': '和尚去参加漫展了', 'output': '他化二次缘了'}


In [17]:
simplified_ruozhi_training

Dataset({
    features: ['raw', 'instruction', 'output'],
    num_rows: 3439
})

## Sampling 50 data points for the many-shot rewriting prompt

In [18]:
rewrite_sample = simplified_ruozhi_training.shuffle(seed=42).select(range(50))

In [19]:
rewrite_sample[0]

{'raw': '路边一公一母狗打架。老头扑上去咬死了公狗。并称是英雄救美。',
 'instruction': '我看到路边有两只狗在打架',
 'output': '有个老头扑上去咬死了公狗，并称是英雄救美。'}

In [20]:
from langchain_core.prompts import ChatPromptTemplate, FewShotChatMessagePromptTemplate
from langchain_core.output_parsers import JsonOutputParser
from pydantic import BaseModel, Field

class RewriteData(BaseModel):
    instruction: str = Field(description="问答形式的问部分")
    output: str = Field(description="问答形式的答部分")

parser = JsonOutputParser(pydantic_object=RewriteData)

example_prompt = ChatPromptTemplate.from_messages([
    ("human", """原段子: {raw}
    
    请分解为问答形式：
    问:
    答:"""),
    ("ai", """问: {instruction}
答: {output}""")
])

many_shot_prompt = FewShotChatMessagePromptTemplate(
    examples=rewrite_sample,
    example_prompt=example_prompt,
)

In [21]:
final_prompt = ChatPromptTemplate.from_messages([
    ("system", "你是一个专业的段子改写助手。你擅长将弱智吧段子分解为一问一答的结构化格式。请严格按照以下json格式回复，不要添加任何额外文本。"),
    many_shot_prompt,
    ("human", "原段子: {raw}\n\n{format_instructions}")
])

In [22]:
from langchain_openai import ChatOpenAI
model = ChatOpenAI(model="gpt-4o-2024-11-20", temperature=0.5)
rewrite_chain = final_prompt | model | parser

In [23]:
new_raw = "我是时间的主人，因为我经常抽时间。"
result = rewrite_chain.invoke({"raw": new_raw, "format_instructions": parser.get_format_instructions()})

In [24]:
result

{'instruction': '为什么说你是时间的主人？', 'output': '因为我经常抽时间。'}

In [25]:
df_2025 = pd.read_csv('ruozhiba_2025_raw.csv')
ruozhi_2025 = df_2025['raw'].to_list()


In [26]:
def process_in_batches(items, batch_size=5, delay=1.0):
    """Process items in small batches with delays"""
    all_instructions = []
    all_outputs = []
    
    for i in tqdm(range(0, len(items), batch_size), desc="Batches"):
        batch = items[i:i+batch_size]
        batch_inputs = [
            {"raw": r, "format_instructions": parser.get_format_instructions()}
            for r in batch
        ]
        
        try:
            batch_results = rewrite_chain.batch(
                batch_inputs,
                config={
                    "max_concurrency": 2,  # Conservative
                    "request_timeout": 60
                }
            )
            
            # Extract results
            for result in batch_results:
                all_instructions.append(result.get("instruction", ""))
                all_outputs.append(result.get("output", ""))
                
        except Exception as e:
            print(f"Batch {i//batch_size + 1} failed: {e}")
            # Add empty results for failed batch
            all_instructions.extend([""] * len(batch))
            all_outputs.extend([""] * len(batch))
        
        time.sleep(delay)  # 1 sec between batches
    
    return all_instructions, all_outputs

In [29]:
instruction_list, output_list = process_in_batches(ruozhi_2025, batch_size=1, delay=10.0)

Batches: 100%|██████████| 139/139 [25:50<00:00, 11.16s/it]


In [30]:
df_2025['instruction'] = instruction_list
df_2025['output'] = output_list
df_2025.to_csv('rewritten_ruozhi_2025.csv')

In [3]:
revised_df_2025 = pd.read_csv('rewritten_ruozhi_2025_manually_revised.csv')
ds_revised = Dataset.from_pandas(revised_df_2025)

In [7]:
HF_USER = "franzyellow"
DATASET_NAME = f"{HF_USER}/ruozhi_2025_test"
ds_revised.push_to_hub(DATASET_NAME, private=True)

Creating parquet from Arrow format: 100%|██████████| 1/1 [00:00<00:00, 174.83ba/s]
Processing Files (1 / 1): 100%|██████████| 33.7kB / 33.7kB, 56.2kB/s  
New Data Upload: 100%|██████████| 33.7kB / 33.7kB, 56.2kB/s  
Uploading the dataset shards: 100%|██████████| 1/1 [00:01<00:00,  1.40s/it]


CommitInfo(commit_url='https://huggingface.co/datasets/franzyellow/ruozhi_2025_test/commit/10e7215daba7f5e747b05309c27d010c2dce55f1', commit_message='Upload dataset', commit_description='', oid='10e7215daba7f5e747b05309c27d010c2dce55f1', pr_url=None, repo_url=RepoUrl('https://huggingface.co/datasets/franzyellow/ruozhi_2025_test', endpoint='https://huggingface.co', repo_type='dataset', repo_id='franzyellow/ruozhi_2025_test'), pr_revision=None, pr_num=None)

## Adjust ruozhiba_punchline for fine-tuning

In [2]:
ruozhi_punchline = load_dataset("LooksJuicy/ruozhiba-punchline", split="train", trust_remote_code=True)

In [3]:
def simplify_ft(example): 
    return { "instruction": example["instruction"], 
            "output": example["output"],
            "text": example["instruction"].strip() + "ANSWER:" + example["output"].strip() } 
    
ruozhi_ft = ruozhi_punchline.map(simplify_ft, remove_columns=ruozhi_punchline.column_names)
print(ruozhi_ft[0])
print(ruozhi_ft[100])

Map: 100%|██████████| 3439/3439 [00:00<00:00, 35364.38 examples/s]

{'instruction': '和尚去参加漫展了', 'output': '他化二次缘了', 'text': '和尚去参加漫展了ANSWER:他化二次缘了'}
{'instruction': '益州有什么特产？', 'output': '众所周知，益州的特产是铅，正所谓天下三分，益州Pb。', 'text': '益州有什么特产？ANSWER:众所周知，益州的特产是铅，正所谓天下三分，益州Pb。'}


In [4]:
HF_USER = "franzyellow"
DATASET_NAME = f"{HF_USER}/ruozhiba_punchline_ft"
ruozhi_ft.push_to_hub(DATASET_NAME, private=True)

Creating parquet from Arrow format: 100%|██████████| 4/4 [00:00<00:00, 705.55ba/s]


Processing Files (1 / 1): 100%|██████████|  573kB /  573kB,  503kB/s  
New Data Upload: 100%|██████████|  503kB /  503kB,  503kB/s  
Uploading the dataset shards: 100%|██████████| 1/1 [00:01<00:00,  1.87s/it]


CommitInfo(commit_url='https://huggingface.co/datasets/franzyellow/ruozhiba_punchline_ft/commit/f2e3ee11b1c95e4d13aa3e26a41d31a448faf232', commit_message='Upload dataset', commit_description='', oid='f2e3ee11b1c95e4d13aa3e26a41d31a448faf232', pr_url=None, repo_url=RepoUrl('https://huggingface.co/datasets/franzyellow/ruozhiba_punchline_ft', endpoint='https://huggingface.co', repo_type='dataset', repo_id='franzyellow/ruozhiba_punchline_ft'), pr_revision=None, pr_num=None)

In [17]:
def simplify_ft(example):
    text = f"用户：{example['instruction']}\n助手：{example['output']}"
    return {
        "text": text
    }

ruozhi_ft = ruozhi_punchline.map(simplify_ft, remove_columns=ruozhi_punchline.column_names)
print(ruozhi_ft[0])
print(ruozhi_ft[100])

{'text': '用户：和尚去参加漫展了\n助手：他化二次缘了'}
{'text': '用户：益州有什么特产？\n助手：众所周知，益州的特产是铅，正所谓天下三分，益州Pb。'}


In [18]:
HF_USER = "franzyellow"
DATASET_NAME = f"{HF_USER}/ruozhiba_punchline_ft_single_domain"
ruozhi_ft.push_to_hub(DATASET_NAME, private=True)

Creating parquet from Arrow format: 100%|██████████| 4/4 [00:00<00:00, 1686.66ba/s]
Processing Files (1 / 1): 100%|██████████|  281kB /  281kB,  281kB/s  
New Data Upload: 100%|██████████|  281kB /  281kB,  281kB/s  
Uploading the dataset shards: 100%|██████████| 1/1 [00:02<00:00,  2.01s/it]


CommitInfo(commit_url='https://huggingface.co/datasets/franzyellow/ruozhiba_punchline_ft_single_domain/commit/e519774cfeca5fdccb7ddca5c165401fda23f54e', commit_message='Upload dataset', commit_description='', oid='e519774cfeca5fdccb7ddca5c165401fda23f54e', pr_url=None, repo_url=RepoUrl('https://huggingface.co/datasets/franzyellow/ruozhiba_punchline_ft_single_domain', endpoint='https://huggingface.co', repo_type='dataset', repo_id='franzyellow/ruozhiba_punchline_ft_single_domain'), pr_revision=None, pr_num=None)

In [19]:
ruozhi_ft[0]

{'text': '用户：和尚去参加漫展了\n助手：他化二次缘了'}